# Notebook 06 — Multi-Query: várias formulações, um ranking só

Terceiro notebook do **pipeline RAG**. O notebook 03 terminou com um limite: uma
pergunta curta como "boleto" traz elogios genéricos, porque a busca só tem uma
palavra para trabalhar. O cliente, porém, escreve "paguei o boleto e o pedido não
foi liberado". O **Multi-Query** da LangChain aula 03 ataca isso: o LLM reescreve a
consulta de vários jeitos, o Buscador roda cada um e o RRF funde os rankings.

| Seção | Tema | Função em `src/voc_rag` | Aula de origem |
|---|---|---|---|
| 0 | Setup | — | — |
| 1 | O problema: uma consulta só | `busca.Buscador` | notebook 03 |
| 2 | Gerar variações com schema Pydantic | `multiquery.criar_gerador_variacoes` | LangChain 03 (video 02) · LangChain 02 |
| 3 | Buscar cada variação e fundir com RRF | `multiquery.buscar_multi_query` | LangChain 03 · IA Gen 02 (slide 24) |
| 4 | Vale o custo? Precisão@10 com e sem Multi-Query | `avaliacao.precisao_em_k` | IA Gen 02 (slides 26–28) |

> Pré-requisitos: notebooks 01 a 05 e o `.env` com a chave do LLM.

In [ ]:
import inspect
import time

import pandas as pd

from voc_rag import avaliacao, busca, indice, llm, multiquery
from voc_rag.preparacao import carregar_base

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)


def mostrar_codigo(objeto):
    """Imprime o código-fonte de uma função ou classe do src/, para lermos junto com o resultado."""
    print(inspect.getsource(objeto))


documentos = carregar_base()
indice_vetorial, _ = indice.carregar_indice()
buscador = busca.Buscador(documentos, indice_vetorial, indice.carregar_modelo())
modelo_chat = llm.criar_llm()
gerador = multiquery.criar_gerador_variacoes(modelo_chat)

## Seção 1 — O problema: uma consulta só

A mesma pergunta do fim do notebook 03. Olhe a coluna `texto`: quantas reviews
falam mesmo de boleto ou pagamento?

In [ ]:
COLUNAS = ["pontuacao", "similaridade", "nota", "texto"]
PADRAO_BOLETO = r"boleto|pagamento|pagu|compens|cobranc|cobrad"

uma_so = buscador.buscar("boleto", k=10)
print(f"precisão@10 = {avaliacao.precisao_em_k(uma_so, PADRAO_BOLETO):.1f}")
uma_so[COLUNAS]

## Seção 2 — Gerar variações *(LangChain aula 03, video 02 · LangChain aula 02)*

Na aula 03 o LLM devolve as variações em texto livre, uma por linha, e um regex
(`_extrair_variacoes`) tira a numeração. Funciona até o dia em que o modelo responde
"Claro! Aqui estão as variações:". Aqui o LLM responde no schema `Variacoes` com
`with_structured_output` — o mesmo padrão do roteador do notebook 05 — e
`limpar_variacoes` só tira repetidas.

O prompt pede que as variações soem como **o cliente escreveria numa review**: a
busca é contra reviews, não contra perguntas de analista. E pede para **não inventar**
produto, lugar ou período: os filtros já foram extraídos pelo roteador.

In [ ]:
mostrar_codigo(multiquery.Variacoes)
print(multiquery.PROMPT.messages[0].prompt.template)

In [ ]:
for consulta in ["boleto", "produto diferente do anunciado", "atendimento do vendedor"]:
    consultas, avisos = multiquery.gerar_consultas(gerador, consulta)
    print(f"{consulta!r}")
    for c in consultas[1:]:
        print(f"   -> {c}")
    print(*avisos)

## Seção 3 — Buscar cada variação e fundir *(LangChain aula 03 · IA Gen aula 02, slide 24)*

A aula 03 junta os documentos de todas as consultas numa lista sem repetidos, na
ordem em que apareceram. Aqui usamos o mesmo **RRF** da busca híbrida, agora entre
as consultas: uma review que aparece bem em várias formulações sobe no ranking.
Duas colunas novas:

- `n_consultas`: em quantas formulações a review apareceu (sinal de consenso);
- `similaridade`: a **maior** similaridade entre as consultas. É o sinal de evidência
  que a etapa 4 vai usar para decidir se recusa.

In [ ]:
mostrar_codigo(multiquery.buscar_multi_query)

In [ ]:
consultas, _ = multiquery.gerar_consultas(gerador, "boleto")
multi = multiquery.buscar_multi_query(buscador, consultas, k=10)
print(f"consultas: {consultas}")
print(f"precisão@10 = {avaliacao.precisao_em_k(multi, PADRAO_BOLETO):.1f} "
      f"(uma consulta só: {avaliacao.precisao_em_k(uma_so, PADRAO_BOLETO):.1f}) | "
      f"{multi.attrs['latencia_ms']:.0f} ms de busca")
multi[["pontuacao", "similaridade", "n_consultas", "nota", "texto"]]

## Seção 4 — Vale o custo? *(IA Gen aula 02, slides 26–28)*

O Multi-Query custa **uma chamada ao LLM** e **N buscas** por pergunta. Para saber se
compensa, rodamos a checagem do notebook 03 (10 perguntas, juiz por regex) com e sem
ele. Lembre o limite da medida: o juiz é lexical, e as variações podem acertar o
vocabulário do regex. Leia as diferenças pergunta a pergunta.

> O `criar_llm` limita as chamadas por minuto (`LLM_RPM` no `.env`), para não estourar
> a cota gratuita do Gemini. Esta célula leva cerca de um minuto e meio.

In [ ]:
linhas = []
for pergunta, padrao in avaliacao.PERGUNTAS_CHECAGEM:
    simples = buscador.buscar(pergunta, k=10)
    inicio = time.perf_counter()
    consultas, _ = multiquery.gerar_consultas(gerador, pergunta)
    tempo_llm = time.perf_counter() - inicio
    multi = multiquery.buscar_multi_query(buscador, consultas, k=10)
    linhas.append({
        "pergunta": pergunta,
        "p@10 híbrida": avaliacao.precisao_em_k(simples, padrao),
        "p@10 multi-query": avaliacao.precisao_em_k(multi, padrao),
        "sim. top1 híbrida": float(simples["similaridade"].max()),
        "sim. top1 multi": float(multi["similaridade"].max()),
        "LLM (s)": round(tempo_llm, 1),
    })
comparacao = pd.DataFrame(linhas)
comparacao

In [ ]:
comparacao[["p@10 híbrida", "p@10 multi-query", "LLM (s)"]].mean().round(2)

## Resumo

- O **Multi-Query** reformula a consulta com o LLM e funde as buscas com RRF; perguntas
  curtas ou vagas são as que mais ganham.
- As variações vêm num **schema Pydantic**, não em texto livre: sem regex para limpar.
- Se o LLM falhar, `gerar_consultas` segue só com a consulta original.
- O custo é uma chamada ao LLM por pergunta; a tabela da Seção 4 mostra se ele se paga.

**Próxima etapa:** checagem de evidência e recusa. Com o top-k do Multi-Query em mãos,
o RAG decide se há evidência suficiente para responder — ou se deve recusar.